# NetSense-QoR — M7 generalisation on Kaggle

Tests how well the models work on **designs they have never seen** (PRD experiment 6, section 8.2).
3 variants (`m7_baseline`, `m7_ratio`, `m7_combined`) × 3 seeds × (1 random-split model + 4 held-out-design fold models)
= 45 training runs, plus few-shot fine-tuning, on 12 designs. Expect roughly **3 hours** on a GPU.

**Setup** (same as before): *Add Input* → your `openabc-mini` dataset; *Settings*: **Accelerator = GPU**, **Internet = on**.

**Run it as a saved version** so it keeps running when you close the tab and the outputs are kept:
*Save Version* → *Save & Run All (Commit)*. When it has finished, open the version and download the notebook
(*File* → *Download notebook*, or the ⋮ menu); the results table, per-run and per-design numbers and the bar chart are stored inside it.


In [ ]:
!git clone https://github.com/tejaswini-18/netsense.git
%cd netsense
!git log --oneline -1

In [ ]:
# Kaggle already has PyTorch (with GPU support), pandas, scipy, matplotlib, tqdm and pyyaml.
# Install only what is missing. Do NOT reinstall torch: that breaks the running session.
!pip install -q torch_geometric pytest
import torch, torch_geometric
print("torch", torch.__version__, "| PyG", torch_geometric.__version__, "| GPU:", torch.cuda.is_available())

In [ ]:
# Find the attached dataset (the folder that contains synthesisStatistics.pickle)
from pathlib import Path
DATA_ROOT = next(Path("/kaggle/input").rglob("synthesisStatistics.pickle")).parent
print("Dataset:", DATA_ROOT, "| design graphs:", len(list((DATA_ROOT / "graphs").glob("*.pt"))), "(expected 42)")

In [ ]:
# Quick check that everything works here before the long run
!NETSENSE_DATA_ROOT={DATA_ROOT} python -m pytest -q 2>&1 | tail -3


In [ ]:
# All variants x seeds x settings. Progress bars are filtered out to keep the saved notebook small;
# each epoch line, each run's summary and a '>>>' line per finished setting are still printed.
!python -m src.run_generalisation --data-root {DATA_ROOT} --runs-dir /kaggle/working/runs --out /kaggle/working/results/generalisation 2>&1 | tr '\r' '\n' | grep -vE 'it/s\]|s/it\]|^ *$'


In [ ]:
# Show the results (also stored inside this notebook when it is downloaded)
from IPython.display import Image, display
out = Path("/kaggle/working/results/generalisation")
print((out / "results_table.md").read_text())
print("===== results_table.csv =====")
print((out / "results_table.csv").read_text())
display(Image(str(out / "generalisation_bar.png")))
